In [1]:
import sys
import torch
import pandas as pd
import numpy as np

print("🎉 MÔI TRƯỜNG THI SẴN SÀNG!")
print(f"- Python Version: {sys.version.split()[0]}")
print(f"- PyTorch Version: {torch.__version__}")
print(f"- GPU (CUDA) Status: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"- GPU Name: {torch.cuda.get_device_name(0)}")

🎉 MÔI TRƯỜNG THI SẴN SÀNG!
- Python Version: 3.10.20
- PyTorch Version: 2.13.0+cpu
- GPU (CUDA) Status: False


In [2]:
import random
import os
import numpy as np
import torch

# 1. Định nghĩa hàm cố định Seed (Bắt buộc cho kỳ thi để đảm bảo kết quả tái tạo được)
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    print(f"✅ Đã cố định Random Seed = {seed}")

seed_everything(42)

# 2. Thao tác cơ bản với PyTorch Tensor
# Tạo một ma trận ngẫu nhiên 3x3
tensor_a = torch.randn(3, 3)
print("\nMa trận Tensor A:")
print(tensor_a)

# Chuyển Tensor sang mảng NumPy và ngược lại
numpy_arr = tensor_a.numpy()
tensor_b = torch.from_numpy(numpy_arr)

print("\nKích thước Tensor A:", tensor_a.shape)
print("Kiểu dữ liệu:", tensor_a.dtype)

✅ Đã cố định Random Seed = 42

Ma trận Tensor A:
tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863],
        [ 2.2082, -0.6380,  0.4617]])

Kích thước Tensor A: torch.Size([3, 3])
Kiểu dữ liệu: torch.float32


In [3]:
import pandas as pd
import numpy as np

# 1. Giả lập một dữ liệu tập Test (gồm ID và các đặc trưng)
data = {
    'id': [101, 102, 103, 104, 105],
    'feature_1': [0.5, 1.2, np.nan, 3.4, 2.1], # Có 1 giá trị bị thiếu (NaN)
    'feature_2': [10, 20, 30, 40, 50]
}
df = pd.DataFrame(data)

print("--- 1. Dữ liệu gốc ---")
print(df)

# 2. Xử lý dữ liệu bị khuyết (Missing Values) bằng trung bình cộng
df['feature_1'] = df['feature_1'].fillna(df['feature_1'].mean())

print("\n--- 2. Dữ liệu sau khi xử lý NaN ---")
print(df)

# 3. Tạo kết quả dự đoán giả lập và lưu ra file .csv chuẩn định dạng thi
fake_predictions = [0, 1, 1, 0, 1]

def create_submission(test_ids, predictions, output_path='submission.csv'):
    df_sub = pd.DataFrame({
        'id': test_ids,
        'predict_label': predictions
    })
    # Quy tắc quan trọng khi nộp bài: index=False và encoding='utf-8'
    df_sub.to_csv(output_path, index=False, encoding='utf-8')
    print(f"\n✅ Đã xuất file thành công tại: {output_path}")

create_submission(df['id'], fake_predictions)

--- 1. Dữ liệu gốc ---
    id  feature_1  feature_2
0  101        0.5         10
1  102        1.2         20
2  103        NaN         30
3  104        3.4         40
4  105        2.1         50

--- 2. Dữ liệu sau khi xử lý NaN ---
    id  feature_1  feature_2
0  101        0.5         10
1  102        1.2         20
2  103        1.8         30
3  104        3.4         40
4  105        2.1         50

✅ Đã xuất file thành công tại: submission.csv


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Tự định nghĩa một lớp Dataset chuẩn PyTorch
class CustomTabularDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = torch.tensor(features, dtype=torch.float32)
        if labels is not None:
            self.labels = torch.tensor(labels, dtype=torch.long)
        else:
            self.labels = None

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.features[idx], self.labels[idx]
        return self.features[idx]

# 2. Giả lập tập dữ liệu (100 mẫu, 5 đặc trưng)
X_dummy = np.random.randn(100, 5)
y_dummy = np.random.randint(0, 2, size=100)

# 3. Khởi tạo Dataset & DataLoader
dataset = CustomTabularDataset(X_dummy, y_dummy)
# Chia dữ liệu thành từng batch 16 mẫu, có xáo trộn (shuffle=True)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# 4. Kiểm tra cách DataLoader xuất dữ liệu theo Batch
print(f"Tổng số mẫu dữ liệu: {len(dataset)}")
print(f"Số lượng Batch (mỗi batch 16 mẫu): {len(dataloader)}\n")

for batch_idx, (inputs, labels) in enumerate(dataloader):
    print(f"Batch {batch_idx + 1}: Kích thước inputs = {inputs.shape}, Kích thước labels = {labels.shape}")

Tổng số mẫu dữ liệu: 100
Số lượng Batch (mỗi batch 16 mẫu): 7

Batch 1: Kích thước inputs = torch.Size([16, 5]), Kích thước labels = torch.Size([16])
Batch 2: Kích thước inputs = torch.Size([16, 5]), Kích thước labels = torch.Size([16])
Batch 3: Kích thước inputs = torch.Size([16, 5]), Kích thước labels = torch.Size([16])
Batch 4: Kích thước inputs = torch.Size([16, 5]), Kích thước labels = torch.Size([16])
Batch 5: Kích thước inputs = torch.Size([16, 5]), Kích thước labels = torch.Size([16])
Batch 6: Kích thước inputs = torch.Size([16, 5]), Kích thước labels = torch.Size([16])
Batch 7: Kích thước inputs = torch.Size([4, 5]), Kích thước labels = torch.Size([4])
